In [ ]:
%%writefile agents.py


from crewai import Agent
import os
from langchain_google_genai import ChatGoogleGenerativeAI

# Get the Gemini API key from environment variable
gemini_api_key = os.environ.get("GOOGLE_API_KEY")

if not gemini_api_key:
    raise ValueError("GOOGLE_API_KEY environment variable not set. Please set it before running.")

# Use ChatGoogleGenerativeAI instance directly
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=gemini_api_key)

hate_speech_detector= Agent(
    role="Hate Speech Detector",
    goal="Accurately detect and identify instances of hate speech, discriminatory language, or harmful content in text.",
    llm=llm,
    backstory=("""you are a hate speech detector for twitter who understands details very well
    and a expet negotiator who can identify hate speech/ offensive language in given tweet"""),
    verbose=True
)

Overwriting agents.py


In [79]:
%%writefile tasks.py


from crewai import Task
from agents import hate_speech_detector


hate_speech_detection_task= Task(description=(
   "Analyze the following text to determine if it contains any hate speech or offensive language."
    "Follow these steps:\n"
    "1. Read the text carefully.\n"
    "2. Identify any language that targets a group or individual based on attributes such as race, gender, ethnicity, religion, sexual orientation, disability, or any other protected characteristic.\n"
    "3. Look for threats, dehumanizing language, insults, or promotion of violence or hatred.\n"
    "4. Evaluate the context to ensure words are not taken out of context. Consider intent and potential impact.\n"
    "5. Make an objective decision based on the content. Provide a clear 'Hate Speech' or 'No hate speech' classification.\n"
    "6. If classified as 'Hate Speech', explain *why* it is considered hate speech, referencing specific parts of the text and the criteria violated. If classified as 'No hate speech', briefly explain *why* it is not.\n"
    "Text:\n{text}"
    ),
expected_output="Provide a classification (Hate Speech / No hate speech) followed by a detailed explanation and reasoning for your decision.",
agent=hate_speech_detector,
)

Overwriting tasks.py


In [80]:
%%writefile main.py


import os
from crewai import Crew

from agents import hate_speech_detector
from tasks import hate_speech_detection_task

crew= Crew(
    agents=[hate_speech_detector],
    tasks=[hate_speech_detection_task]
    )


while True:
    Text = input("Enter the text to analyze (type 'quit' or 'exit' to stop): ")
    if Text.lower() == 'quit' or Text.lower() == 'exit':
        print("Exiting analysis.")
        break

    result = crew.kickoff(inputs={"text": Text})

    print("Response: ", result)

    # Write results to a Markdown file
    with open("results.md", "a") as f:
        f.write(f"## Analysis of: \"{Text}\"\n")
        f.write(f"### Result:\n{result}\n\n")
        f.write("---\n\n") # Separator for readability

Overwriting main.py


In [78]:
from google.colab import userdata
import os

# Get the API key from Colab secrets
api_key = userdata.get("GOOGLE_API_KEY")

# Set it as an environment variable for the script execution
os.environ["GOOGLE_API_KEY"] = api_key

# Ensure no OpenAI API key is set to prevent conflicts
if "OPENAI_API_KEY" in os.environ:
    del os.environ["OPENAI_API_KEY"]

# Now run the main.py script
get_ipython().system('python main.py')

india is dirty
Enter the text to analyze (type 'quit' or 'exit' to stop): 

> Entering new CrewAgentExecutor chain...
Thought: I have carefully read the text and analyzed it according to the provided steps. I have considered the potential target group, the nature of the language used, and the context and potential impact. I have made an objective decision and am prepared to provide a detailed explanation. I now can give a great answer
Final Answer: Hate Speech.

**Explanation and Reasoning:**
The statement "india is dirty" is classified as Hate Speech.

1.  **Targeted Group:** The statement targets an entire country, "India," and by direct implication, its people and culture. Nationality and national origin are recognized as protected characteristics under many anti-discrimination and hate speech policies. Generalizing negatively about an entire nation inherently reflects upon its inhabitants as a group.
2.  **Derogatory and Insulting Language:** The word "dirty" in this context is not